# STEF Stage 1 & Stage 2/3-lite — Real Diagnostics
## AKK Corridor (Nigeria) and DMIC Corridor (India) vs. Rhine–Ruhr, Germany

This is the canonical, tested version of this notebook — matching exactly what
produced the results reported in the book chapter (Tables 4–8). Earlier versions
used OpenStreetMap (via Overpass API) for the Label Scarcity Score proxy; that
approach is blocked by a network restriction in Kaggle's environment, so this
version uses a GEE-native alternative (VIIRS nighttime-lights radiance) instead.


In [ ]:
!pip install -q earthengine-api geemap scikit-learn
import ee, numpy as np, json
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score


## 1. Authenticate Earth Engine (interactive)

1. Register a free Earth Engine project at https://code.earthengine.google.com/register
2. Run the cell below — it prints a URL; open it, sign in, approve, paste the
   verification code back here.
3. Set `PROJECT_ID` to your own project ID.


In [ ]:
PROJECT_ID = "warm-torus-420318"  # replace with your own Earth Engine project ID

ee.Authenticate(auth_mode='notebook', force=True)
ee.Initialize(project=PROJECT_ID)
print("Earth Engine initialised.")


## 2. Define source and both target corridors

- **Source**: Rhine–Ruhr industrial region, Germany (real European reference domain).
- **Target 1**: AKK gas corridor, Nigeria — waypoints traced from a published
  pipeline map (Ajaokuta → Baro → Suleja → Kaduna → Zaria → Kano).
- **Target 2**: DMIC corridor, India — waypoints traced from published DMIC/DFC
  alignment maps (Dadri → Alwar → Jaipur → Ahmedabad/Gandhinagar → Dholera SIR →
  Surat → JNPT/Mumbai).

Neither route is the exact as-built centreline (not public) — both are real,
map-traced approximations, not straight lines between endpoint cities.


In [ ]:
source_aoi = ee.Geometry.Rectangle([6.8, 51.3, 7.6, 51.6])  # Rhine-Ruhr, Germany

akk_pts = ee.Geometry.LineString([
    [6.6667, 7.5000],   # Ajaokuta
    [6.7500, 8.5500],   # Baro
    [7.1806, 9.1817],   # Suleja
    [7.4383, 10.5222],  # Kaduna
    [7.7000, 11.0667],  # Zaria
    [8.5920, 12.0022],  # Kano
])
target_aoi = akk_pts.buffer(5000)  # AKK corridor, Nigeria

dmic_pts = ee.Geometry.LineString([
    [77.5537, 28.5546],  # Dadri (Delhi NCR)
    [76.6346, 27.5530],  # Alwar
    [75.7873, 26.9124],  # Jaipur
    [72.5714, 23.0225],  # Ahmedabad / Gandhinagar
    [72.1833, 22.2500],  # Dholera SIR
    [72.8311, 21.1702],  # Surat
    [72.9525, 18.9490],  # JNPT / Mumbai
])
dmic_aoi = dmic_pts.buffer(5000)  # DMIC corridor, India

print("AKK (Nigeria) area, km²:", target_aoi.area().divide(1e6).getInfo())
print("DMIC (India) area, km²:", dmic_aoi.area().divide(1e6).getInfo())
print("Source (Rhine-Ruhr) area, km²:", source_aoi.area().divide(1e6).getInfo())


## 3. Sentinel-2 composites, SSI, LSS (VIIRS), III — reusable functions

Defined once, applied to both target regions.


In [ ]:
def s2_composite(aoi, year=2025):
    def mask_clouds(img):
        scl = img.select("SCL")
        mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
        return img.updateMask(mask)
    coll = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(aoi)
            .filterDate(f"{year}-01-01", f"{year}-12-31")
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
            .map(mask_clouds))
    return coll.median().clip(aoi)

def sample_band(img, aoi, band, n=2000, scale=20):
    fc = img.select(band).sample(region=aoi, scale=scale, numPixels=n, seed=42, geometries=False)
    vals = fc.aggregate_array(band).getInfo()
    return np.array([v for v in vals if v is not None], dtype=float)

def bhattacharyya_ssi(a, b, bins=50):
    lo, hi = min(a.min(), b.min()), max(a.max(), b.max())
    pa, _ = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    pb, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    pa = pa / pa.sum() if pa.sum() > 0 else pa
    pb = pb / pb.sum() if pb.sum() > 0 else pb
    bc = np.clip(np.sum(np.sqrt(pa * pb)), 1e-12, 1.0)
    return 1 - np.exp(np.log(bc))

BANDS = {"B5": "Red-Edge", "B11": "SWIR1"}

def compute_ssi(source_img, source_aoi, target_img, target_aoi):
    per_band = {}
    for b in BANDS:
        s = sample_band(source_img, source_aoi, b)
        t = sample_band(target_img, target_aoi, b)
        per_band[b] = bhattacharyya_ssi(s, t)
    return float(np.mean(list(per_band.values()))), per_band

# LSS proxy: VIIRS nighttime-lights radiance ratio (GEE-native — avoids the
# Overpass/OSM network dependency, which is blocked in this environment).
viirs = (ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
         .filterDate("2023-01-01", "2024-01-01").select("avg_rad").mean())

def mean_radiance(aoi):
    stats = viirs.reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=500, maxPixels=1e9).getInfo()
    return stats.get("avg_rad") or 0

def compute_lss(source_rad, target_rad):
    return float(np.clip(1 - (target_rad / (source_rad + 1e-9)), 0, 1))

def compute_iii(aoi):
    worldpop = (ee.ImageCollection("WorldPop/GP/100m/pop")
                .filterBounds(aoi).filterDate("2020-01-01", "2021-01-01").mosaic().clip(aoi))
    stats = worldpop.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=aoi, scale=100, maxPixels=1e9).getInfo()
    mean_pop, std_pop = stats.get("population_mean") or 0, stats.get("population_stdDev") or 0
    cv_raw = std_pop / (mean_pop + 1e-9)
    return float(1 - np.exp(-cv_raw)), cv_raw

def classify_pathway(ssi, lss):
    if ssi < 0.2 and lss < 0.4:
        return "A — Zero-shot GeoFM embedding"
    elif ssi < 0.5 and lss < 0.75:
        return "B — Parameter-efficient fine-tuning (LoRA)"
    else:
        return "C — Domain adversarial adaptation"


## 4. Nigeria (AKK corridor) — Stage 1

In [ ]:
target_img = s2_composite(target_aoi)
source_img = s2_composite(source_aoi)

SSI, ssi_per_band = compute_ssi(source_img, source_aoi, target_img, target_aoi)
source_rad = mean_radiance(source_aoi)
target_rad_akk = mean_radiance(target_aoi)
LSS = compute_lss(source_rad, target_rad_akk)
III, iii_cv = compute_iii(target_aoi)
pathway = classify_pathway(SSI, LSS)

stage1_nigeria = {
    "region": "AKK corridor, Nigeria", "SSI": round(SSI, 3),
    "SSI_per_band": {k: round(v, 3) for k, v in ssi_per_band.items()},
    "LSS_proxy": round(LSS, 3), "III_proxy": round(III, 3),
    "recommended_pathway": pathway,
}
print(json.dumps(stage1_nigeria, indent=2))
with open("stef_stage1_nigeria_results.json", "w") as f:
    json.dump(stage1_nigeria, f, indent=2)


## 5. Nigeria — Stage 2/3-lite (Random Forest vs. ESA WorldCover)

In [ ]:
FEATURE_BANDS = ["B2", "B3", "B4", "B5", "B8", "B11", "B12"]
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
MIN_PER_CLASS = 15

def sample_with_labels(img, aoi, n=800, scale=30, seed=1):
    stacked = img.select(FEATURE_BANDS).addBands(worldcover.rename("label"))
    fc = stacked.sample(region=aoi, scale=scale, numPixels=n, seed=seed, geometries=False, tileScale=8)
    recs = fc.getInfo()["features"]
    X, y = [], []
    for r in recs:
        p = r["properties"]
        if p.get("label") is None or any(p.get(b) is None for b in FEATURE_BANDS):
            continue
        X.append([p[b] for b in FEATURE_BANDS]); y.append(p["label"])
    return np.array(X, dtype=float), np.array(y, dtype=int)

X_source, y_source = sample_with_labels(source_img, source_aoi, seed=1)
X_target, y_target = sample_with_labels(target_img, target_aoi, seed=2)

src_counts, tgt_counts = Counter(y_source), Counter(y_target)
common_classes = sorted([c for c in src_counts if src_counts[c] >= MIN_PER_CLASS and tgt_counts.get(c, 0) >= MIN_PER_CLASS])
print("Classes used:", common_classes)

def filter_classes(X, y, classes):
    mask = np.isin(y, classes)
    return X[mask], y[mask]

X_source, y_source = filter_classes(X_source, y_source, common_classes)
X_target, y_target = filter_classes(X_target, y_target, common_classes)


In [ ]:
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(X_source, y_source, test_size=0.3, random_state=0, stratify=y_source)
clf_base = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xs_tr, ys_tr)
f1_indomain = f1_score(ys_te, clf_base.predict(Xs_te), average="macro")

clf_zs = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_source, y_source)
f1_zeroshot = f1_score(y_target, clf_zs.predict(X_target), average="macro")

Xt_adapt, Xt_hold, yt_adapt, yt_hold = train_test_split(
    X_target, y_target, train_size=min(50, len(y_target) - 10), random_state=0, stratify=y_target)
clf_ad = RandomForestClassifier(n_estimators=200, random_state=0).fit(
    np.vstack([X_source, Xt_adapt]), np.concatenate([y_source, yt_adapt]))
f1_adapted = f1_score(yt_hold, clf_ad.predict(Xt_hold), average="macro")

stage23_nigeria = {
    "region": "AKK corridor, Nigeria",
    "f1_indomain_source": round(float(f1_indomain), 3),
    "f1_zeroshot_target": round(float(f1_zeroshot), 3),
    "f1_adapted_target": round(float(f1_adapted), 3),
    "n_target_labels_used_for_adaptation": int(len(yt_adapt)),
    "classes_used": [int(c) for c in common_classes],
}
print(json.dumps(stage23_nigeria, indent=2))
with open("stef_stage23_nigeria_results.json", "w") as f:
    json.dump(stage23_nigeria, f, indent=2)


## 6. India (DMIC corridor) — Stage 1

In [ ]:
dmic_img = s2_composite(dmic_aoi)

SSI_dmic, ssi_per_band_dmic = compute_ssi(source_img, source_aoi, dmic_img, dmic_aoi)
target_rad_dmic = mean_radiance(dmic_aoi)
LSS_dmic = compute_lss(source_rad, target_rad_dmic)
III_dmic, iii_cv_dmic = compute_iii(dmic_aoi)
pathway_dmic = classify_pathway(SSI_dmic, LSS_dmic)

stage1_india = {
    "region": "DMIC corridor, India", "SSI": round(SSI_dmic, 3),
    "SSI_per_band": {k: round(v, 3) for k, v in ssi_per_band_dmic.items()},
    "LSS_proxy": round(LSS_dmic, 3), "III_proxy": round(III_dmic, 3),
    "recommended_pathway": pathway_dmic,
}
print(json.dumps(stage1_india, indent=2))
with open("stef_stage1_india_results.json", "w") as f:
    json.dump(stage1_india, f, indent=2)


## 7. India — Stage 2/3-lite (reuses Nigeria's source-domain training data)

In [ ]:
X_dmic, y_dmic = sample_with_labels(dmic_img, dmic_aoi, seed=3)
dmic_counts = Counter(y_dmic)
common_dmic = sorted([c for c in src_counts if src_counts[c] >= MIN_PER_CLASS and dmic_counts.get(c, 0) >= MIN_PER_CLASS])
print("Classes used (India):", common_dmic)
X_dmic_f, y_dmic_f = filter_classes(X_dmic, y_dmic, common_dmic)

mask_eval = np.isin(y_dmic_f, clf_zs.classes_)
f1_zeroshot_dmic = f1_score(y_dmic_f[mask_eval], clf_zs.predict(X_dmic_f[mask_eval]), average="macro")

Xd_adapt, Xd_hold, yd_adapt, yd_hold = train_test_split(
    X_dmic_f, y_dmic_f, train_size=min(50, len(y_dmic_f) - 10), random_state=0, stratify=y_dmic_f)
clf_ad_dmic = RandomForestClassifier(n_estimators=200, random_state=0).fit(
    np.vstack([X_source, Xd_adapt]), np.concatenate([y_source, yd_adapt]))
f1_adapted_dmic = f1_score(yd_hold, clf_ad_dmic.predict(Xd_hold), average="macro")

stage23_india = {
    "region": "DMIC corridor, India",
    "f1_indomain_source": round(float(f1_indomain), 3),
    "f1_zeroshot_target": round(float(f1_zeroshot_dmic), 3),
    "f1_adapted_target": round(float(f1_adapted_dmic), 3),
    "n_target_labels_used_for_adaptation": int(len(yd_adapt)),
    "classes_used": [int(c) for c in common_dmic],
}
print(json.dumps(stage23_india, indent=2))
with open("stef_stage23_india_results.json", "w") as f:
    json.dump(stage23_india, f, indent=2)
